# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_before.png" style="width:500px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [2]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env(".env")

OPENAI_API_KEY=****8FMA
LANGSMITH_API_KEY=****573f
LANGCHAIN_TRACING_V2=true
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****pocs
LANGSMITH_ENDPOINT=****.com


In [3]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [4]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [5]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [6]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [8]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_MXzDYqcWOCKcsEfsqs1h5ojR)
 Call ID: call_MXzDYqcWOCKcsEfsqs1h5ojR
  Args:
    query: SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_5TgDObXKJEqIj3RNgrvIB7Wm)
 Call ID: call_5TgDObXKJEqIj3RNgrvIB7Wm
  Args:
    query: SELECT name, sql FROM sqlite_master WHERE type = 'table' AND name IN ('Customer','Invoice') ORDER BY name;
================================= Tool Message =================================
Name: execute_sql

[('Customer

In [9]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

Could you clarify which titles you mean—titles of what, and any filters (e.g., date range, author, category)? If you’re referring to titles from an earlier result, please paste or describe that context.


## Add memory

In [20]:
from langgraph.checkpoint.memory import InMemorySaver

In [21]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [22]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_8B5bf7bDVksSqxwHIVSLYzHL)
 Call ID: call_8B5bf7bDVksSqxwHIVSLYzHL
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' AND (name LIKE '%invoice%' OR name LIKE '%bill%' OR name LIKE '%customer%' OR name LIKE '%client%') ORDER BY name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Customer',), ('Invoice',), ('InvoiceLine',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_E6B35G89IhJnsa7WNu8AZKcM)
 Call ID: call_E6B35G89IhJnsa7WNu8AZKcM
  Args:
    query: SELECT sql FROM sqlite_master WHERE type='table' AND name='Customer' LIMIT 5;
================================= Tool Message ===============

In [23]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_pHNrZLLKXoLgp2DU0YwAPQut)
 Call ID: call_pHNrZLLKXoLgp2DU0YwAPQut
  Args:
    query: SELECT t.Name AS Title
FROM InvoiceLine AS il
JOIN Track AS t ON t.TrackId = il.TrackId
WHERE il.InvoiceId = (
  SELECT i.InvoiceId
  FROM Invoice AS i
  JOIN Customer AS c ON c.CustomerId = i.CustomerId
  WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
  ORDER BY i.InvoiceDate DESC, i.InvoiceId DESC
  LIMIT 1
)
ORDER BY il.InvoiceLineId
LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Holier Than Thou',), ('Through The Never',), ('My Friend Of Misery',), ('The Wait',), ('Blitzkrieg',)]
================================== Ai Message ==================================

Here are the titles on your most recent invoice (first 

## Try your own queries
Now that there is memory, check the agents recall!

In [24]:
question = "How many invoices do I have?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

How many invoices do I have?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_hcTqFWVyu3CwtP7R5ROABjAF)
 Call ID: call_hcTqFWVyu3CwtP7R5ROABjAF
  Args:
    query: SELECT COUNT(*) AS InvoiceCount
FROM Invoice AS i
JOIN Customer AS c ON c.CustomerId = i.CustomerId
WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris';
================================= Tool Message =================================
Name: execute_sql

[(7,)]
================================== Ai Message ==================================

You have 7 invoices.
